# RQ4 - Fairness & Ethics: Algorithmic Bias across School Tracks

### Context :

In Swiss canton Zurich, students can enter gymnasium either after 6th primary school (Langzeitgymnasium, 6 years total) or after 2nd/3rd secondary school (Kurzzeitgymnasium, 4 years total). Both tracks lead to the same Maturität certificate and university access, but Langzeit students tend to come from more academically selected and socioeconomically advantaged backgrounds, having passed a competitive entrance exam at a younger age, while Kurzzeitgymnasium draws from a broader demographic.

### Hypothesis :

Does the predictive model perform equally well for both tracks? If models perform systematically better for Langzeit students, this could reflect algorithmic bias, amplifying existing socioeconomic inequalities by providing less reliable automated feedback to already disadvantaged students.

### Structure : 

**Part 1 - Essay Performance (RQ1) Fairness** (`essay_results.csv`). We reuse the RF model from RQ1 and evaluate whether prediction quality differs between tracks on essay coherence scoring.


**Part 2 - Chatbot Engagement (RQ2) Fairness** (`gymitrainer.csv`). We reuse the RF model from RQ2 and assess fairness along two dimensions: (A) descriptive differences in chatbot usage between tracks, and (B) per-group prediction performance (MAE, R², bias) via 5-fold cross-validation.

**Part 3 - Early Behavior (RQ3) Fairness** (early-window behavioral features). We reuse the Gradient Boosting model from RQ3 and evaluate whether early learning behaviors predict student success equally well across tracks, using per-group AUC and fairness metrics (FNR, FPR, TPR, disparate impact).

In [106]:
# Standard library
import ast
import math
import os
import sys

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Scikit-learn
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import confusion_matrix, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# IPython
from IPython.display import display

# Local
import helpers
import importlib
from Q3.utils import *

In [87]:
# Load data directory from environment or config.py, fallback to ../data
# sys.path.append('..') is already set above, so config is importable
DATA_DIR = os.environ.get('GOGYMI_DATA')
if not DATA_DIR:
    try:
        from config import get_data_dir
        DATA_DIR = get_data_dir()
    except Exception:
        if os.path.isdir('../data'):
            DATA_DIR = '../data'
        else:
            raise RuntimeError('Set GOGYMI_DATA or create config.py from config_example.py or provide ../data')
data_dir = DATA_DIR
fig_dir = '.'
print("Using data_dir:", os.path.abspath(data_dir))
print("Figures will be saved in:", os.path.abspath(fig_dir))

Using data_dir: c:\Users\msgar\OneDrive\Documents\EPFL\Cours\MA4\ML4BD\MLBD_2026\data
Figures will be saved in: c:\Users\msgar\OneDrive\Documents\EPFL\Cours\MA4\ML4BD\MLBD_2026\Q4_OverallFairness


---

## Part 1 - Essay Performance (RQ1) Fairness

In [88]:
essay_results = pd.read_csv(f'{data_dir}/essay_results.csv')

### Step 1.1 - Track Distribution and Score Analysis

In the GoGymi platform, essay courses map to two school id's that correspond to the two different school tracks :

| course_id | Track | Type |
|---|---|---|
| 5447 | Langzeitgymnasium | Essay |
| 3301 | Kurzzeitgymnasium | Essay |

We first examine whether the two tracks differ in essay volume, score distributions, and writing characteristics, before evaluating whether our model treats them equally.

In [89]:
essay_df = essay_results.copy()
essay_df["track"] = essay_df["course"].map({5447: "Langzeit", 3301: "Kurzzeitgymnasium"})

print(essay_df["track"].value_counts())
print()
print(essay_df.groupby("track")["structure__coherence"].agg(["mean", "std", "count"]))

track
Langzeit             6336
Kurzzeitgymnasium    3859
Name: count, dtype: int64

                       mean       std  count
track                                       
Kurzzeitgymnasium  7.292563  1.484558   3859
Langzeit           7.255447  1.633220   6334


The dataset contains **6,336 Langzeit** and **3,859 Kurzzeitgymnasium** essays, which is roughly a 60/40 split, reflecting the larger enrollment of the longer track.

Mean `structure_coherence` scores are nearly identical across tracks (Langzeit: 7.26, Kurzzeitgymnasium: 7.29), suggesting the AI scorer does not systematically favor either group at the population level. However, Langzeit essays show slightly higher variance (std=1.63 vs 1.48), indicating more spread in writing quality, consistent with a more heterogeneous student population entering the program earlier.

These baseline differences motivate the core fairness question: even if average scores are similar, does the model predict individual scores equally reliably for both groups?

### Step 1.2 - Feature Engineering (same as RQ1)

In [90]:
essay_df = essay_df.dropna(subset=["course", "text_type", "text", "structure__coherence"]).copy()

essay_df["word_count"] = essay_df["text"].apply(helpers.word_count)
essay_df["char_count"] = essay_df["text"].apply(helpers.char_count)
essay_df["sentence_count"] = essay_df["text"].apply(helpers.sentence_count)
essay_df["avg_sentence_length"] = essay_df["text"].apply(helpers.avg_sentence_length)
essay_df["avg_word_length"] = essay_df["text"].apply(helpers.avg_word_length)
essay_df["lexical_diversity"] = essay_df["text"].apply(helpers.lexical_diversity)
essay_df["comma_count"] = essay_df["text"].apply(helpers.comma_count)
essay_df["semicolon_count"] = essay_df["text"].apply(helpers.semicolon_count)
essay_df["colon_count"] = essay_df["text"].apply(helpers.colon_count)
essay_df["exclamation_count"] = essay_df["text"].apply(helpers.exclamation_count)
essay_df["question_count"] = essay_df["text"].apply(helpers.question_count)
essay_df["long_word_ratio"] = essay_df["text"].apply(helpers.long_word_ratio)

feature_cols = [
    "course", "text_type", "word_count", "char_count", "sentence_count",
    "avg_sentence_length", "avg_word_length", "lexical_diversity",
    "comma_count", "semicolon_count", "colon_count", "exclamation_count",
    "question_count", "long_word_ratio"
]

cat_cols = ["course", "text_type"]
num_cols = [c for c in feature_cols if c not in cat_cols]

X = essay_df[feature_cols].copy()
y = essay_df["structure__coherence"].copy()
track = essay_df["track"].copy()

### Step 1.3 - Per-group fairness evaluation

Train on full data, evaluate per track using OOF predictions

In [91]:
preprocessor = ColumnTransformer(transformers=[
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

rf_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_oof = cross_val_predict(rf_model, X, y, cv=cv, n_jobs=-1)

# per-group metrics
from sklearn.metrics import mean_absolute_error, r2_score

results = []
for t in ["Langzeit", "Kurzzeitgymnasium"]:
    mask = track == t
    mae = mean_absolute_error(y[mask], y_pred_oof[mask])
    r2 = r2_score(y[mask], y_pred_oof[mask])
    results.append({"Track": t, "N": mask.sum(), "MAE": round(mae, 4), "R²": round(r2, 4)})

df_fairness = pd.DataFrame(results).set_index("Track")
df_fairness.head()

,N,MAE,R²
Track,,,
Langzeit,6239,0.9807,0.3336
Kurzzeitgymnasium,3826,0.9486,0.2712


In [92]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, track_name in zip(axes, ["Langzeit", "Kurzzeitgymnasium"]):
    mask = track == track_name
    ax.scatter(y[mask], y_pred_oof[mask], alpha=0.3, s=10)
    lims = [y[mask].min(), y[mask].max()]
    ax.plot(lims, lims, "r--")
    ax.set_title(f"{track_name}\nMAE={df_fairness.loc[track_name, 'MAE']:.3f}, R²={df_fairness.loc[track_name, 'R²']:.3f}")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")

plt.suptitle("Fairness — Per-track prediction performance (RF, OOF)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fairness_per_track_scatter.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

Both tracks show the same prediction pattern, no visual difference between them. The R² gap (0.33 vs 0.27) is real but not visible to the naked eye at this scale.

In [93]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, track_name in zip(axes, ["Langzeit", "Kurzzeitgymnasium"]):
    mask = track == track_name
    ax.hist(y[mask], bins=11, range=(0,10), edgecolor="black")
    ax.set_title(f"{track_name} — score distribution")
    ax.set_xlabel("structure__coherence")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(f"{fig_dir}/fairness_score_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

Almost identical between tracks, both peak at 7-8, and we can see the same shape. The AI scorer treats both tracks the same way. Nothing interesting here.

In [94]:
residuals = y.values - y_pred_oof
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, track_name in zip(axes, ["Langzeit", "Kurzzeitgymnasium"]):
    mask = (track == track_name).values
    ax.hist(residuals[mask], bins=30, edgecolor="black")
    ax.axvline(0, color="red", linestyle="--")
    ax.set_title(f"{track_name} — residual distribution")
    ax.set_xlabel("Residual (actual - predicted)")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(f"{fig_dir}/fairness_residuals.png", dpi=300, bbox_inches="tight")
plt.show()

Both symmetric and centered around 0. We can't see systematic over or under-prediction for either track. 

In [95]:
features_to_plot = ["word_count", "lexical_diversity", "avg_word_length", "avg_sentence_length"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, features_to_plot):
    for track_name, color in zip(["Langzeit", "Kurzzeitgymnasium"], ["#4393c3", "#d73027"]):
        mask = essay_df["track"] == track_name
        ax.hist(essay_df.loc[mask, feat], bins=30, alpha=0.5, label=track_name, color=color)
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle("Feature distributions by track", fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fairness_feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

word_count: Langzeit essays are longer on average, which makes sense since it's longer program.

avg_word_length and avg_sentence_length are virtually identical.

### Step 1.4 - Results and Interpretation

| Track | N | MAE | R² |
|---|---|---|---|
| Langzeit | 6239 | 0.981 | 0.334 |
| Kurzzeitgymnasium | 3826 | 0.949 | 0.271 |

The model performs differently across the two tracks, though not dramatically. MAE is slightly lower for Kurzzeitgymnasium (0.95 vs 0.98), suggesting marginally smaller absolute errors. However R² is notably higher for Langzeit (0.33 vs 0.27), meaning the model explains more variance for that group.

This discrepancy is likely driven by score variance rather than true bias: Langzeit essays show higher standard deviation (1.63 vs 1.48), giving the model more signal to learn from. Kurzzeitgymnasium essays cluster more tightly around the mean, making individual predictions harder to differentiate.

Importantly, the scatter plots reveal that the AI scorer assigns discrete integer scores, producing the vertical banding visible in both plots. This quantization effect limits prediction precision regardless of track, and may itself introduce a form of scoring bias by forcing continuous writing quality into discrete bins.

**Fairness conclusion:** No strong evidence of systematic algorithmic bias against either track. However, the model is less informative for Kurzzeitgymnasium students (lower R²), meaning automated feedback based on this model would be less slightly reliable for that group, which warrants attention given the potential socioeconomic differences between tracks. We tested for bias and found none, at least for the essay part.

---

## Part 2 - Chatbot Engagement (RQ2) Fairness

This section analyses whether a Random Forest model predicting student performance from GymiTrainer chatbot engagement features treats Langzeit and Kurzzeit students equally. The analysis is carried over from R2 and addresses two questions:

#### A - Descriptive analysis: Do the two tracks engage with the chatbot differently?  

#### B - Predictive fairness: Does the model predict performance equally well for both tracks?

### Step 2.1 - Preparation of datasets, features, and performance score, as in RQ2.

In [96]:
students        = pd.read_csv(f'{data_dir}/students.csv')
teachers        = pd.read_csv(f'{data_dir}/teachers.csv')
quiz_res        = pd.read_csv(f'{data_dir}/quiz_results.csv')
text_res        = pd.read_csv(f'{data_dir}/text_results.csv')
essay_res       = pd.read_csv(f'{data_dir}/essay_results.csv')
gymitrainer     = pd.read_csv(f'{data_dir}/gymitrainer.csv')
gymitrainer_feedback = pd.read_csv(f'{data_dir}/gymitrainer_feedback.csv')

In [97]:
# Performance score
quiz_res['norm_score'] = quiz_res['points'] / quiz_res['max_points']
quiz_score_p3 = quiz_res.groupby('user_id')['norm_score'].mean().rename('quiz_score')

essay_dim_cols = [
    'content__on_topic', 'content__plausible', 'content__convincing_ideas',
    'content__scope', 'content__correct',
    'structure__coherence', 'structure__outline', 'structure__repetition',
    'structure__text_pattern',
    'language__clarity', 'language__spelling', 'language__puncutation',
    'language__word_choice', 'language__sentence_structure', 'language__style'
]
essay_res['mean_dim_score'] = essay_res[essay_dim_cols].mean(axis=1, skipna=True)
essay_score_p3 = essay_res.groupby('user_id')['mean_dim_score'].mean().rename('essay_score')

performance_p3 = pd.concat([quiz_score_p3, essay_score_p3], axis=1)
performance_p3['essay_score_norm'] = (performance_p3['essay_score'] / 6).clip(0, 1)
performance_p3['performance_score'] = performance_p3[['quiz_score', 'essay_score_norm']].mean(axis=1, skipna=True)
teachers_users_p3 = set(teachers['user_id'])
performance_clean_p3 = performance_p3[~performance_p3.index.isin(teachers_users_p3)]

# Chatbot features
def parse_content(content_str):
    try:
        messages = ast.literal_eval(content_str)
        n_messages = len(messages)
        n_student = sum(1 for m in messages if not m.get('gymitrainer', True))
        n_bot = sum(1 for m in messages if m.get('gymitrainer', False))
        times = [m['time'] for m in messages if 'time' in m]
        duration = (max(times) - min(times)) if len(times) > 1 else 0
        return pd.Series([n_messages, n_student, n_bot, duration])
    except:
        return pd.Series([None, None, None, None])

gymitrainer[['n_messages', 'n_student_messages', 'n_bot_messages', 'session_duration_s']] = (
    gymitrainer['content'].apply(parse_content)
)
chatbot_freq = (
    gymitrainer.groupby('user_id')
    .agg(n_sessions=('Unnamed: 0', 'count'), n_unique_pages=('url', 'nunique'))
    .reset_index()
)
chatbot_intensity = (
    gymitrainer.groupby('user_id')
    .agg(
        avg_messages_per_session=('n_messages', 'mean'),
        avg_student_messages=('n_student_messages', 'mean'),
        avg_bot_messages=('n_bot_messages', 'mean'),
        total_messages=('n_messages', 'sum'),
        avg_session_duration_s=('session_duration_s', 'median'),
        max_messages_session=('n_messages', 'max'),
    )
    .reset_index()
)
feedback_with_user = gymitrainer_feedback.merge(
    gymitrainer[['Unnamed: 0', 'user_id']], left_on='thread_id', right_on='Unnamed: 0', how='left'
)
chatbot_feedback = (
    feedback_with_user.groupby('user_id')
    .agg(n_feedbacks_given=('score', 'count'), avg_feedback_score=('score', 'mean'))
    .reset_index()
)
chatbot_features = (
    chatbot_freq
    .merge(chatbot_intensity, on='user_id', how='outer')
    .merge(chatbot_feedback, on='user_id', how='outer')
)
sq2_data = chatbot_features.merge(
    performance_clean_p3[['performance_score']], left_on='user_id', right_index=True, how='inner'
)
sq2_data['n_feedbacks_given'] = sq2_data['n_feedbacks_given'].fillna(0)
sq2_data['avg_feedback_score'] = sq2_data['avg_feedback_score'].fillna(0)
print(f"sq2_data: {sq2_data.shape}")

sq2_data: (1033, 12)


### Step 2.2 - **A** - Descriptive analysis: do the two groups engage with the chatbot differently?

Before assessing model fairness, we first examine whether Langzeit and Kurzzeit students exhibit different chatbot usage patterns. If the two groups behave differently on the platform, this has direct implications for how well a model trained on the full population can generalise to each group individually.

Since the GoGymi dataset contains no explicit track variable, we derive each student's track (Langzeitgymnasium or Kurzzeitgymnasium) from the course IDs present in their quiz, text comprehension, and essay results, using the mapping provided in the data description  
(Langzeit: 42, 5447, 2115 - Kurzzeit: 3865, 3301, 5009).

In [98]:
LANGZEIT_IDS = {42, 5447, 2115}
KURZZEIT_IDS = {3865, 3301, 5009}

def assign_track(course_set):
    has_lang = bool(course_set & LANGZEIT_IDS)
    has_kurz = bool(course_set & KURZZEIT_IDS)
    if has_lang and not has_kurz:
        return 'Langzeit'
    elif has_kurz and not has_lang:
        return 'Kurzzeit'
    if course_set & {42} and not course_set & {3865}:
        return 'Langzeit'
    elif course_set & {3865} and not course_set & {42}:
        return 'Kurzzeit'
    n_lang = len(course_set & LANGZEIT_IDS)
    n_kurz = len(course_set & KURZZEIT_IDS)
    if n_lang > n_kurz:
        return 'Langzeit'
    elif n_kurz > n_lang:
        return 'Kurzzeit'
    return None

course_per_user = (
    pd.concat([
        quiz_res[['user_id', 'course_id']],
        text_res[['user_id', 'course_id']],
        essay_res.rename(columns={'course': 'course_id'})[['user_id', 'course_id']],
    ])
    .dropna(subset=['course_id'])
    .astype({'course_id': int})
    .groupby('user_id')['course_id']
    .apply(set)
    .reset_index()
    .rename(columns={'course_id': 'course_set'})
)

course_per_user['track'] = course_per_user['course_set'].apply(assign_track)
track_labels = course_per_user[['user_id', 'track']].dropna(subset=['track'])

print(track_labels['track'].value_counts())
print(f"\nUsers with ambiguous/unknown track dropped: {course_per_user['track'].isna().sum()}")

track
Kurzzeit    1062
Langzeit     994
Name: count, dtype: int64

Users with ambiguous/unknown track dropped: 91


Most students appeared exclusively in one track's course IDs and were assigned directly. For the 275 students with mixed course IDs (exercises attempted across both tracks), we applied a two-step resolution: first prioritising the Math course ID (42 or 3865) as the strongest signal, then falling back to a majority vote over all course IDs present. This resolved the majority of mixed cases. The remaining 91 students were perfect ties, for which no principled assignment could be made; these are excluded from the fairness analysis, representing approximately 4% of the labeled population.

The final analysis population consists of **994 Langzeit** and **1,062 Kurzzeit** students.

In [99]:
# Merge track into sq2_data

sq2_fair = sq2_data.merge(track_labels, on='user_id', how='inner')

print(f"sq2_fair shape: {sq2_fair.shape}")
print(sq2_fair['track'].value_counts())
print(f"\nStudents dropped (no track label): {len(sq2_data) - len(sq2_fair)}")

sq2_fair shape: (1026, 13)
track
Kurzzeit    592
Langzeit    434
Name: count, dtype: int64

Students dropped (no track label): 7


After merging with the track labels, the fairness analysis population consists of **1,026 students** (592 Kurzzeit, 434 Langzeit). This is the subset of students who both used the chatbot and have a performance score.  
The modest class imbalance (58% / 42%) is noted but does not require rebalancing for this analysis, as we report per-group metrics independently rather than training a classifier on track membership.

In [100]:
# Descriptive plot: do the two groups use the chatbot differently?

engagement_cols = [
    'n_sessions', 'n_unique_pages',
    'avg_messages_per_session', 'avg_student_messages',
    'avg_session_duration_s', 'max_messages_session',
    'n_feedbacks_given', 'avg_feedback_score'
]

COLORS = {'Langzeit': 'steelblue', 'Kurzzeit': 'tomato'}

print("Median chatbot engagement by track:")
print(sq2_fair.groupby('track')[engagement_cols].median().T.round(3))

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for ax, col in zip(axes, engagement_cols):
    data = [sq2_fair[sq2_fair['track'] == t][col].dropna() for t in ['Langzeit', 'Kurzzeit']]
    bp = ax.boxplot(data, patch_artist=True, tick_labels=['Langzeit', 'Kurzzeit'])
    for patch, t in zip(bp['boxes'], ['Langzeit', 'Kurzzeit']):
        patch.set_facecolor(COLORS[t])
    ax.set_title(col, fontsize=9)

plt.suptitle('Chatbot engagement by track: Langzeit vs Kurzzeit', fontsize=12)
plt.tight_layout()
plt.savefig(f'{fig_dir}/2_chatbot_engagement_by_track.png', dpi=150, bbox_inches='tight')
plt.show()

Median chatbot engagement by track:
track                     Kurzzeit  Langzeit
n_sessions                   6.000     5.500
n_unique_pages               4.000     4.000
avg_messages_per_session     7.727     7.000
avg_student_messages         4.000     3.667
avg_session_duration_s     147.250    93.000
max_messages_session        17.500    15.000
n_feedbacks_given            0.000     0.000
avg_feedback_score           0.000     0.000


**Results of the Descriptive Analysis (A)**

The median chatbot engagement metrics are remarkably similar between the two tracks across most dimensions. Both groups have nearly identical session counts (6.0 vs 5.5), unique pages visited (4.0 vs 4.0), and message depth (avg_messages_per_session: 7.7 vs 7.0, avg_student_messages: 4.0 vs 3.7).

The most notable difference is in session duration: Kurzzeit students spend a median of 147s per session versus 93s for Langzeit students, suggesting Kurzzeit students tend to engage more deeply per session despite similar frequency.  
The feedback columns (n_feedbacks_given, avg_feedback_score) have a median of 0 for both groups, confirming that feedback rating is a rare behaviour regardless of track.

The boxplots reveal that the two distributions are highly overlapping across all features, with both groups showing heavy right-skewed distributions and large outliers. This pattern is consistent with the full population observed in the chatbot feature engineering step.  
The spread is slightly wider for Langzeit students in `avg_session_duration_s` (one extreme outlier near 125,000s), but this does not affect the median comparison.

Overall, the descriptive analysis suggests that track membership does not strongly determine how students use the chatbot, which motivates the next question: even if usage patterns are similar, does the model predict performance equally well for both groups?

### Step 2.3 - **B** - Fairness of performance prediction across tracks

We then evaluate whether the Random Forest model trained on chatbot features predicts student performance equally well for both groups. Specifically, we assess per-group prediction error (MAE), explained variance ($R^2$), and systematic bias (mean signed error). A model that is systematically less accurate or more biased for Kurzzeit students, who on average come from less privileged backgrounds, would represent an equity concern worth surfacing, as such a model could reinforce existing disparities if used to guide educational interventions.

In [101]:
feature_cols_fair = [c for c in sq2_fair.columns
                     if c not in ['user_id', 'performance_score', 'track', 'course_set']]

X_fair     = sq2_fair[feature_cols_fair].fillna(0)
y_fair     = sq2_fair['performance_score']
track_fair = sq2_fair['track']

rf_fair = RandomForestRegressor(n_estimators=100, random_state=42)
y_pred  = cross_val_predict(rf_fair, X_fair, y_fair, cv=5)

results_fair = pd.DataFrame({
    'user_id':   sq2_fair['user_id'].values,
    'track':     track_fair.values,
    'y_true':    y_fair.values,
    'y_pred':    y_pred,
    'error':     y_pred - y_fair.values,
    'abs_error': np.abs(y_pred - y_fair.values)
})

print(f"{'Track':10s} │ {'n':>4} │ {'MAE':>6} │ {'R²':>6} │ {'Bias':>6}")
print("─" * 45)
for track_name, grp in results_fair.groupby('track'):
    mae  = mean_absolute_error(grp['y_true'], grp['y_pred'])
    r2   = r2_score(grp['y_true'], grp['y_pred'])
    bias = grp['error'].mean()
    print(f"{track_name:10s} │ {len(grp):>4} │ {mae:>6.3f} │ {r2:>6.3f} │ {bias:>+6.3f}")

Track      │    n │    MAE │     R² │   Bias
─────────────────────────────────────────────
Kurzzeit   │  592 │  0.107 │ -0.012 │ +0.010
Langzeit   │  434 │  0.114 │ -0.135 │ -0.018


Both tracks yield negative $R^2$ values, confirming that the model fails to explain performance variance for either group. Kurzzeit students are predicted slightly more accurately (MAE=0.107, $R^2$=−0.012) than Langzeit students (MAE=0.114, $R^2$=−0.135). Bias is near zero for both groups (+0.010 and −0.018 respectively), indicating no systematic over- or under-prediction.

In [102]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# a) Actual performance score by track
sq2_fair.boxplot(column='performance_score', by='track', ax=axes[0])
axes[0].set_title('Actual performance score by track')
axes[0].set_xlabel('Track')
axes[0].set_ylabel('Performance score')

# b) Prediction error distribution by track
for track_name, grp in results_fair.groupby('track'):
    axes[1].hist(grp['error'], bins=20, alpha=0.6, label=track_name, density=True)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Prediction error distribution\n(positive = over-predicted)')
axes[1].set_xlabel('Predicted − Actual')
axes[1].legend()

# c) Predicted vs actual, coloured by track
for track_name, grp in results_fair.groupby('track'):
    axes[2].scatter(grp['y_true'], grp['y_pred'], alpha=0.4, label=track_name, s=15)
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[2].set_title('Predicted vs Actual performance')
axes[2].set_xlabel('Actual')
axes[2].set_ylabel('Predicted')
axes[2].legend()

plt.suptitle('Fairness analysis: Langzeit vs Kurzzeit', fontsize=13)
plt.tight_layout()
plt.savefig(f'{fig_dir}/2_fairness_chatbot_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

- The boxplot on the left shows that actual performance distributions are nearly identical between tracks, with both centred around 0.65-0.70 and a similar spread. This confirms that the groups are comparable in outcome. 
- The error distribution (centre) is centred near zero for both tracks but is notably wider for Langzeit students, meaning predictions are less consistent for that group.
- The predicted vs actual scatter (right) shows the model compresses predictions toward the mean (0.6-0.7 range) for both tracks, struggling equally to capture low-performing students.

### Step 2.4 - Results and Interpretation

The fairness analysis shows no evidence of systematic bias against either track. Errors are small and near zero for both groups. However, the model is uniformly poor for both Langzeit and Kurzzeit students, which is itself a fairness concern: an unreliable model applied in an educational context would be equally unhelpful to all students regardless of track. The key finding is the model's general inability to predict performance from chatbot features alone, a limitation that affects both tracks equally.  

This reinforces the conclusion from Part 5: chatbot engagement reflects student struggle rather than driving performance outcomes, and this pattern holds regardless of the track the students belong to.

---

## Part 3 - Early Behavior (RQ3) Fairness

This question examines whether the early-behaviour classifier performs equitably across the two Swiss gymnasium tracks, Langzeitgymnasium and Kurzzeitgymnasium, which differ in socioeconomic composition. 

Per-group metrics including AUC, false negative rate, and false positive rate are computed to detect whether one track is systematically disadvantaged by the model's predictions.

### Step 3.1 - Load and visualize features

Vizualisation of the feature space using KMeans and prediction with clusters as a feature.

Prior work has shown that learners naturally segment into a small number of behaviorally distinct profiles.(Kizilcec, R. F., Piech, https://doi.org/10.1145/2460296.2460330). They established that unsupervised clustering makes subpopulation appear.

We apply K-Means (k=3) to our feature space and project the results onto the first two principal components to assess separability.

In [103]:
# Load Features from Q3
df_final = pd.read_csv(r"q3_features.csv")
feature_cols = [c for c in df_final.columns if c not in ("user_id", "label")]

In [104]:
X = df_final[feature_cols].fillna(0) # Handle missing values
y = df_final["label"].astype(int)
X = X.astype('float64')
# 2. Standardize the data
scaler = StandardScaler()
groups = {
    "clicks":    ["clicks__n_total", "clicks__session_gap_cv", "clicks__scroll_depth_max"],
    "exercise":  ["unique_quiz_qids", "unique_math_qids", "unique_text_qids", "q__revisit_rate"],
    "content":   ["content_entropy", "q__n_unique_courses"],
    "media":     ["media__n_play_events", "media__completion_rate", "media__audio_pct"],
}

scaled_parts = []
for name, cols in groups.items():
    available = [c for c in cols if c in df_final.columns]
    scaled_parts.append(pd.DataFrame(
        scaler.fit_transform(df_final[available].fillna(0)),
        columns=available
    ))


X_scaled = pd.concat(scaled_parts, axis=1)


kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_final['cluster'] = clusters


pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df_final['pca1'] = pca_results[:, 0]
df_final['pca2'] = pca_results[:, 1]


plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='pca1', y='pca2', 
    hue='cluster', 
    palette='viridis', 
    data=df_final, 
    s=100, alpha=0.7
)

plt.title('Student Behavioral Profiles: Clustering by Early Learning Metrics')
plt.xlabel('Principal Component 1 ')
plt.ylabel('Principal Component 2')
plt.legend(title='Student Group')
plt.savefig("PCA.jpg")

We observe three profiles:

- A large group (Yellow) clustered around the origin
- A blue group spread along the positive PC1 axis.
- A small group (Purple) isolated in the positive region.

Let's dig more into what these clusters mean about student behavior.

### Step 3.2 - Clustering profiles

In [105]:
# Plot clustering profiles 
key_features = [
    "unique_quiz_qids", "unique_math_qids", "q__n_active_days",
    "q__n_questions_viewed", "content_entropy",
    "clicks__session_gap_cv", "clicks__n_total", "media__n_unique_urls", "q__revisit_rate", "media__completion_rate"
]

label_map = {
    "unique_quiz_qids":       "Quiz diversity",
    "unique_math_qids":       "Math diversity",
    "q__n_active_days":       "Active days",
    "q__n_questions_viewed":  "Questions viewed",
    "content_entropy":        "Content entropy",
    "clicks__session_gap_cv": "Session gap CV",
    "clicks__n_total":        "Total clicks",
    "media__n_unique_urls":   "Unique media URLs",
    "q__revisit_rate":        "Revisit rate",
    "media__completion_rate": "Media completion",
}
cluster_labels = {0: "Disengaged", 1: "Engaged", 2: "Moderate"}

key_features = list(label_map.keys())
cluster_means = df_final.groupby("cluster")[key_features].mean()
cluster_means_norm = (cluster_means - cluster_means.mean()) / cluster_means.std()

display = cluster_means_norm.rename(index=cluster_labels, columns=label_map)
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(display.T, annot=True, fmt=".2f",
    cmap="coolwarm", center=0.5,
    linewidths=0.4, linecolor="white",
    cbar_kws={"shrink": 0.75, "label": "z-score"}, 
    ax=ax
         )
plt.title("Cluster Profiles")
plt.xlabel("Cluster")
plt.savefig("cluster_profile.png")

This profiling gives interesting insights of the behavior of the students in each clusters.

- Group 0 is bellow average on everything, reflecting disengaged students, that show low activity, low diversity and low regularity in the beginning.
- Group 1 show highly engaged students accross all dimensions.
- Group 2 is a cluster of moderate users.

In [54]:
print("------------------ Outcome Rate ------------------")
print(df_final.groupby("cluster").agg(
    n_students   = ("label", "count"),
    success_rate = ("label", "mean"),
    pct_langzeit = ("q__maj_track", "mean")
).round(3))

------------------ Outcome Rate ------------------
         n_students  success_rate  pct_langzeit
cluster                                        
0               240         0.512         0.429
1               302         0.460         0.341
2                33         0.667         1.000


Surprisingly, the group of disengaged students seem to have the highest success rate, while the moderate ones have the lowest one. One interesting finding here is that the cluster success rate ordering mirrors the Langzeit representation ordering exactly, suggesting that behavioral engagement on this platform is reflects partially the track of the students. This raises a fundamental question: does our model predict learning success from behavior, or does it predict track membership, which correlates with success through socioeconomic mechanisms outside the platform?

### Step 3.3 - Stratified KFold

In [107]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = build_models()["Gradient Boosting"]

kurz_mask = (df_final['q__maj_track'] == 0).values
lang_mask = (df_final['q__maj_track'] == 1).values

kurz_aucs, lang_aucs = [], []

oof_proba = np.zeros(len(X)) 

for train_idx, test_idx in cv.split(X, y):
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_prob = pipe.predict_proba(X.iloc[test_idx])[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = y.iloc[test_idx].values

    oof_proba[test_idx] = y_prob
    # Which test indices belong to each track
    kurz_fold = kurz_mask[test_idx]
    lang_fold = lang_mask[test_idx]

    if kurz_fold.any():
        kurz_aucs.append(roc_auc_score(y_true[kurz_fold], y_prob[kurz_fold]))
        
    if lang_fold.any():
        lang_aucs.append(roc_auc_score(y_true[lang_fold], y_prob[lang_fold]))
        

df_final["pred_proba"] = oof_proba
df_final["pred_label"] = (oof_proba >= 0.5).astype(int)

print(f"Kurzeit — AUC: {np.mean(kurz_aucs):.3f}")
print(f"Langzeit — AUC: {np.mean(lang_aucs):.3f}")

Kurzeit — AUC: 0.714
Langzeit — AUC: 0.597


### Step 3.4 - Fairness metrics

In [56]:
def fairness_metrics(df_kurz, df_lang):
    
    # Confusion Matrix
    cm_kurz = confusion_matrix(df_kurz['label'],df_kurz['pred_label'])
    cm_lang = confusion_matrix(df_lang['label'],df_lang['pred_label'])
    TN_k, FP_k, FN_k, TP_k = cm_kurz.ravel()
    TN_l, FP_l, FN_l, TP_l = cm_lang.ravel()

    
    # Total population
    N_k = TP_k + FP_k + FN_k + TN_k
    N_l = TP_l + FP_l + FN_l + TN_l 

    
    # True positive rate
    tpr_k = TP_k / (TP_k + FN_k) 
    tpr_l = TP_l / (TP_l + FN_l) 


    # Percentage predicted as positive
    ppp_k = (TP_k+FP_k)/N_k
    ppp_l = (TP_l+FP_l)/N_l


    # FNR
    fnr_k = FN_k/(FN_k+TP_k)
    fnr_l = FN_l/(FN_l+TP_l)


    # FPR
    fpr_k = FP_k/(FP_k+TN_k)
    fpr_l = FP_l/(FP_l+TN_l)


    metrics = {
    "TPR : Equal Opportunity":  (tpr_k, tpr_l),
    "FPR : False Positive Rate":   (fpr_k, fpr_l),
    "PPP : Disparate Impact": (ppp_k, ppp_l),
    "FNR : False Negative Rate" : (fnr_k, fnr_l)
    }

    return metrics

In [108]:
metrics = fairness_metrics(df_final[df_final['q__maj_track'] == 0], df_final[df_final['q__maj_track'] == 1])
rows = []
for name, (k, l) in metrics.items():
    rows.append({"Metric": name, "Kurzeit": k, "Langzeit": l, "Gap": abs(k - l)})

results_df = pd.DataFrame(rows).set_index("Metric")
results_df.style\
    .format({"Kurzeit": "{:.3f}", "Langzeit": "{:.3f}", "Gap": "{:.3f}"})\
    .set_caption("Fairness Metrics by Track")

,Kurzeit,Langzeit,Gap
Metric,,,
TPR : Equal Opportunity,0.663,0.484,0.179
FPR : False Positive Rate,0.372,0.371,0.001
PPP : Disparate Impact,0.521,0.422,0.099
FNR : False Negative Rate,0.337,0.516,0.179


In [109]:
print(f"PPP ratio = {0.427/0.542:.2f}")

PPP ratio = 0.79


### Step 3.5 - Results and Interpretation

The fairness analysis for sub-question 3 reveals the most pro- nounced disparities. Using the Gradient Boosting classifier evaluated via stratified 5-fold cross-validation, the model achieves AUC = 0.715 for Kurzzeit students but only AUC = 0.587 for Langzeit students, a gap of 0.128. Per-group fairness metrics show a false negative rate (FNR) of 0.314 for Kurzzeit versus 0.505 for Langzeit (gap = 0.191), meaning the model correctly identifies successful Kurzzeit students far more reliably than Langzeit ones. The true positive rate gap mirrors this finding (TPR: 0.686 vs. 0.495). The false positive rate is comparable across groups (0.390 vs. 0.371, gap = 0.020), indicating that the asymmetry is specific to the prediction of success rather than struggle. The disparate impact ratio (PPP ratio = 0.79) falls below the commonly used 80% threshold, suggesting a meaningful disparity in positive predictions across tracks.

---

## Overall Conclusion of Fairness research question

**RQ1:** The essay coherence model shows no systematic bias across tracks. Both Langzeit and Kurzzeit students have similar mean coherence scores
(7.26 vs 7.29) and symmetric residual distributions, with the modest R² gap (0.334 vs 0.271) attributable to differences in score variance
rather than directional bias.

**RQ2:** The chatbot engagement model fails equally for both tracks, with near-zero bias (+0.010 for Kurzzeit, -0.018 for Langzeit) and comparable
MAE (0.107 vs 0.114). No track is disproportionately harmed.

**RQ3:** The early-behaviour classifier shows the most pronounced disparity. Langzeit students have a substantially higher false negative rate (0.505
vs 0.314), meaning successful Langzeit students are far more likely to be misclassified as struggling. The AUC gap of 0.128 and PPP ratio of 0.79
fall below the 80% fairness threshold, indicating meaningful inequity. 

**Overall:** Fairness concerns are absent in RQ1 and RQ2, but RQ3 raises a significant equity concern: early behavioural features are less informative
predictors of success for Langzeit students, suggesting that platform engagement patterns do not capture academic potential equally across tracks.